# Анализ результатов Reactive Mesh AuthZ

Этот notebook предназначен для локального разбора экспериментальных артефактов проекта:

- полного набора `3 x 3 x 30` CSV-результатов;
- smoke-артефактов `reactive` и `baseline`;
- итогового summary JSON;
- snapshot-артефактов из `verify-mvp`.

Notebook не пытается заменить CLI-скрипты из `experiments/`, а даёт проверяемый интерактивный путь для thesis/report-уровня анализа и визуализации.

## Что требуется для запуска

Предполагается Python-окружение с `pandas` и `matplotlib`. Если Jupyter ещё не установлен:

```bash
python3 -m pip install notebook pandas matplotlib
jupyter notebook
```

Открывать notebook нужно из корня репозитория, чтобы относительные пути к `experiments/results` работали без дополнительных настроек.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RESULTS = ROOT / 'experiments' / 'results'
VERIFY = RESULTS / 'verify-mvp'

assert RESULTS.exists(), f'Не найден каталог результатов: {RESULTS}'
RESULTS

: 

In [ ]:
matrix_files = sorted(
    path for path in RESULTS.glob('*.csv')
    if path.name.startswith(('reactive-', 'baseline-ext_authz-', 'baseline-poll-'))
)

matrix_files

In [ ]:
frames = []
for path in matrix_files:
    df = pd.read_csv(path)
    df['source_file'] = path.name
    frames.append(df)

matrix_df = pd.concat(frames, ignore_index=True)
matrix_df.head()

In [ ]:
summary = (
    matrix_df.groupby(['mode'])
    .agg(
        runs=('mode', 'count'),
        latency_mean_ms=('latency_to_enforce_ms', 'mean'),
        latency_p95_ms=('latency_to_enforce_ms', lambda s: s.quantile(0.95)),
        latency_p99_ms=('latency_to_enforce_ms', lambda s: s.quantile(0.99)),
        risk_window_mean=('risk_window_messages', 'mean'),
        post_revoke_deny_rate=('post_revoke_deny', 'mean')
    )
    .reset_index()
)

summary

In [ ]:
profile_summary = (
    matrix_df.groupby(['mode', 'profile'])
    .agg(
        runs=('mode', 'count'),
        latency_p50_ms=('latency_to_enforce_ms', 'median'),
        latency_p95_ms=('latency_to_enforce_ms', lambda s: s.quantile(0.95)),
        risk_window_mean=('risk_window_messages', 'mean'),
        still_running_rate=('still_running_after_observe', 'mean')
    )
    .reset_index()
)

profile_summary

In [ ]:
plot_df = profile_summary.pivot(index='profile', columns='mode', values='risk_window_mean')
ax = plot_df.plot(kind='bar', figsize=(10, 5), grid=True, title='Среднее окно риска по режимам и профилям')
ax.set_ylabel('Сообщения после revoke')
plt.tight_layout()

In [ ]:
reactive_latency = matrix_df[matrix_df['mode'] == 'reactive']
profiles = ['low', 'medium', 'high']
data = [reactive_latency[reactive_latency['profile'] == p]['latency_to_enforce_ms'].dropna() for p in profiles]

plt.figure(figsize=(9, 5))
plt.boxplot(data, labels=profiles, showfliers=True)
plt.title('Latency-to-enforce для reactive режима')
plt.ylabel('мс')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()

In [ ]:
reactive_smoke = pd.read_csv(RESULTS / 'reactive-smoke.csv')
baseline_smoke = json.loads((RESULTS / 'baseline-smoke.json').read_text(encoding='utf-8'))
verify_summary = json.loads((VERIFY / 'summary.json').read_text(encoding='utf-8')) if (VERIFY / 'summary.json').exists() else {}

reactive_smoke, baseline_smoke, verify_summary

## Интерпретация

Практический смысл notebook-а:

- быстро показать разницу между `reactive`, `baseline-ext_authz` и `baseline-poll`;
- отдельно просмотреть окно риска `Δ` в сообщениях;
- использовать те же checked-in CSV/JSON-артефакты, которые лежат в репозитории и участвуют в `verify-mvp`;
- готовить thesis/report-графики без ручного переписывания данных.

Если нужен paper-grade анализ, дальше можно добавить bootstrap-сравнение, ECDF и дополнительные графики прямо в этот notebook.